# MIA round-trip on a scratch DDPM

One SDEdit round-trip as a membership-inference attack score (`pixel-MSE(x0, roundtrip(x0))`), against a small UNet2DModel DDPM trained from scratch on CIFAR-10. Source of the numbers: the user's own research project `~/claude_code/test/mia_inference_loop`, `RESULTS.md` §§14–16 (2026-09-10) — single-shot AUROC ≈ 0.996 at n_member=500, falling to ≈ 0.57 at n_member=20000. **Single seed** — see `README.md`.

Needs `torch torchvision diffusers scikit-learn matplotlib`. Imports model code from the research repo via `sys.path` — nothing is copied or retrained. Checkpoints must already exist under `results/small_ddpm_ckpt_cache/`.

Outputs (written to `out/`): `member_before/after.png`, `nonmember_before/after.png`, `roc_loglog.png`, `auroc_vs_n.png`, `results.json`.

Exercise: vary `--t-edit`; try iterating the loop for `T_max > 1` and watch AUROC *decay* instead of grow (the "iterating doesn't help" negative result, RESULTS.md H4).

In [ ]:
"""Single-shot SDEdit round-trip as a membership-inference attack on a scratch DDPM.

One pass x1 = DDIM-denoise(add_noise(x0, t_edit)) against a small UNet2DModel
trained from scratch on n_member CIFAR-10 images; pixel-MSE(x0, x1) is the
per-sample attack score (Carlini et al., IEEE S&P 2022: log-log ROC, TPR@low-FPR).

Imports the model/training/round-trip code from the user's own research repo
via sys.path (no copies) — see README.md for the exact modules and the member
permutation this script reproduces bit-for-bit.
"""
from __future__ import annotations
import argparse, json, sys
from pathlib import Path

import numpy as np
import torch
import torchvision
from torchvision import transforms
from torchvision.utils import save_image
from torch.utils.data import Subset
from diffusers import DDIMScheduler
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESEARCH_REPO = Path.home() / "claude_code" / "test" / "mia_inference_loop"
sys.path.insert(0, str(RESEARCH_REPO))
from experiments.member_vs_nonmember_ddpm_cifar_scratch import (   # noqa: E402
    train_or_load_small_ddpm, DDIM_STEPS, EPOCHS_DEFAULT, NUM_TRAIN_TIMESTEPS,
)
from experiments.ddpm_cifar_common import sdedit_loop_with_snapshots          # noqa: E402

TEAL, SLATE, RED = "#0f766e", "#475569", "#dc2626"
TEST_SEED = 1234

In [ ]:
# ---- core (<=24 lines): split, round-trip, score, AUROC/TPR --------------

def split(ds_train, ds_test, n_member, per_group):
    g = torch.Generator().manual_seed(0)
    member_idx = torch.randperm(len(ds_train), generator=g).tolist()[:n_member]
    member_x = torch.stack([ds_train[i][0] for i in member_idx[:per_group]])
    g_te = torch.Generator().manual_seed(TEST_SEED)
    nonmember_idx = torch.randperm(len(ds_test), generator=g_te).tolist()[:per_group]
    nonmember_x = torch.stack([ds_test[i][0] for i in nonmember_idx])
    return member_idx, member_x, nonmember_idx, nonmember_x

In [ ]:
def roundtrip(unet, ddim, x0, t_edit, seed):
    snaps = sdedit_loop_with_snapshots(
        unet, ddim, x0, T_max=1, snapshots=[0, 1], t_edit_frac=t_edit,
        ddim_steps=DDIM_STEPS, seed=seed, batch_size=128)
    return snaps[1]

In [ ]:
def mse(a, b):
    return ((a - b) ** 2).flatten(1).mean(1).numpy()

In [ ]:
def auroc_tpr(mse_member, mse_nonmember):
    y = np.r_[np.ones(len(mse_member)), np.zeros(len(mse_nonmember))]
    s = np.r_[mse_member, mse_nonmember]
    auc = roc_auc_score(y, s)
    flip = auc < 0.5
    auc = max(auc, 1 - auc)
    fpr, tpr, _ = roc_curve(y, -s if flip else s)
    tpr1 = float(np.interp(0.01, fpr, tpr))
    tpr5 = float(np.interp(0.05, fpr, tpr))
    return auc, tpr1, tpr5, fpr, tpr

In [ ]:
# ---- plotting --------------------------------------------------------------

def _style(ax):
    ax.set_facecolor("white")
    ax.grid(False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

In [ ]:
def unnormalize(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

In [ ]:
def save_grid(x, path, n=7):
    save_image(unnormalize(x[:n]), path, nrow=n, padding=2)

In [ ]:
def plot_roc_loglog(curves: dict, out_path: Path):
    fig, ax = plt.subplots(figsize=(6, 4.2), dpi=100)
    colors = {500: TEAL, 10000: RED}
    for n, (fpr, tpr, auc, tpr1) in curves.items():
        fpr_c = np.clip(fpr, 1e-3, 1.0)
        ax.plot(fpr_c, tpr, color=colors.get(n, SLATE), linewidth=2.2,
                label=f"n={n} (AUROC={auc:.3f})")
        ax.scatter([0.01], [tpr1], color=colors.get(n, SLATE), zorder=5, s=35)
        ax.annotate(f"TPR@1%FPR={tpr1:.2f}", (0.01, tpr1),
                    textcoords="offset points", xytext=(6, -10), fontsize=11,
                    color=colors.get(n, SLATE))
    ax.plot([1e-3, 1], [1e-3, 1], color=SLATE, linestyle="--", linewidth=1, alpha=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1e-3, 1); ax.set_ylim(1e-3, 1)
    ax.set_xlabel("False positive rate", fontsize=14)
    ax.set_ylabel("True positive rate", fontsize=14)
    ax.set_title("Membership inference ROC (log-log)", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.legend(fontsize=11, loc="lower right", frameon=False)
    _style(ax)
    fig.tight_layout()
    fig.savefig(out_path, facecolor="white")
    plt.close(fig)

In [ ]:
def plot_auroc_vs_n(probe_dir: Path, out_path: Path):
    ns, aurocs = [], []
    for p in sorted(probe_dir.glob("n*_s0.json")):
        d = json.loads(p.read_text())
        ns.append(d["n_member"])
        aurocs.append(d["stage_A_single_shot"]["auroc_pixel_mse"])
    order = np.argsort(ns)
    ns = np.array(ns)[order]; aurocs = np.array(aurocs)[order]
    fig, ax = plt.subplots(figsize=(6, 4.2), dpi=100)
    ax.plot(ns, aurocs, marker="o", color=TEAL, linewidth=2.2, markersize=6)
    ax.axhline(0.5, color=SLATE, linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("n_member (training-set size)", fontsize=14)
    ax.set_ylabel("Single-shot AUROC (pixel-MSE)", fontsize=14)
    ax.set_title("Memorization gate: AUROC vs. n_member (seed 0)", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.set_ylim(0.45, 1.02)
    _style(ax)
    fig.tight_layout()
    fig.savefig(out_path, facecolor="white")
    plt.close(fig)

In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt-dir", default=str(RESEARCH_REPO / "results" / "small_ddpm_ckpt_cache"))
    ap.add_argument("--data-root", default=str(RESEARCH_REPO / "data"))
    ap.add_argument("--probe-dir", default=str(RESEARCH_REPO / "results" / "mechanism_probe_ddpm"))
    ap.add_argument("--n", type=int, nargs="+", default=[500, 10000])
    ap.add_argument("--per-group", type=int, default=500)
    ap.add_argument("--t-edit", type=float, default=0.25)
    ap.add_argument("--epochs", type=int, default=EPOCHS_DEFAULT)
    ap.add_argument("--out", default="out")
    args = ap.parse_args()

    ckpt_dir = Path(args.ckpt_dir)
    assert ckpt_dir.exists(), f"checkpoint dir not found: {ckpt_dir}"
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)

    tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
    ds_train = torchvision.datasets.CIFAR10(args.data_root, train=True, download=True, transform=tfm)
    ds_test = torchvision.datasets.CIFAR10(args.data_root, train=False, download=True, transform=tfm)
    ddim = DDIMScheduler(num_train_timesteps=NUM_TRAIN_TIMESTEPS, beta_schedule="linear")

    results = {}
    curves = {}
    example_data = None
    for n_member in args.n:
        member_idx, member_x, nonmember_idx, nonmember_x = split(ds_train, ds_test, n_member, args.per_group)
        members = Subset(ds_train, member_idx)
        unet, _ = train_or_load_small_ddpm(members, seed=0, device="cpu", epochs=args.epochs)
        # train_or_load_small_ddpm looks for its cache under CACHE_DIR (fixed
        # relative to the research repo) — verify it actually found ckpt_dir.
        m1 = roundtrip(unet, ddim, member_x, args.t_edit, seed=0)
        n1 = roundtrip(unet, ddim, nonmember_x, args.t_edit, seed=n_member + 1)
        mse_m, mse_n = mse(m1, member_x), mse(n1, nonmember_x)
        auc, tpr1, tpr5, fpr, tpr = auroc_tpr(mse_m, mse_n)
        results[str(n_member)] = {
            "auroc": auc, "tpr_at_1pct": tpr1, "tpr_at_5pct": tpr5,
            "mse_member_mean": float(mse_m.mean()), "mse_nonmember_mean": float(mse_n.mean()),
            "per_group": args.per_group,
        }
        curves[n_member] = (fpr, tpr, auc, tpr1)
        print(f"n={n_member}  AUROC={auc:.4f}  TPR@1%FPR={tpr1:.3f}  TPR@5%FPR={tpr5:.3f}  "
              f"mse_m={mse_m.mean():.4g}  mse_n={mse_n.mean():.4g}")

        if n_member == min(args.n):
            member_pick = int(np.argmin(mse_m))
            median_rank = np.argsort(mse_n)[len(mse_n) // 2]
            nonmember_pick = int(median_rank)
            save_grid(member_x[member_pick:member_pick + 7], out / "member_before.png")
            save_grid(m1[member_pick:member_pick + 7], out / "member_after.png")
            save_grid(nonmember_x[nonmember_pick:nonmember_pick + 7], out / "nonmember_before.png")
            save_grid(n1[nonmember_pick:nonmember_pick + 7], out / "nonmember_after.png")
            example_data = {
                "member_idx": member_idx[member_pick:member_pick + 7],
                "nonmember_idx": nonmember_idx[nonmember_pick:nonmember_pick + 7],
                "mse_member": float(mse_m[member_pick]),
                "mse_nonmember": float(mse_n[nonmember_pick]),
            }

    plot_roc_loglog(curves, out / "roc_loglog.png")
    plot_auroc_vs_n(Path(args.probe_dir), out / "auroc_vs_n.png")
    (out / "results.json").write_text(json.dumps({"n": results, "examples": example_data,
                                                    "t_edit": args.t_edit, "ddim_steps": DDIM_STEPS,
                                                    "epochs": args.epochs}, indent=2))
    print(f"wrote results + assets to {out}")

In [ ]:
if __name__ == "__main__":
    main()